# VirtualiZarr S3 Example

This notebook demonstrates the core VirtualiZarr workflow using publicly available NetCDF files on AWS S3:

1. Open a single virtual dataset from S3
2. Combine multiple files with `open_virtual_mfdataset`
3. Write virtual references to an Icechunk store

**Data:** [NASA NEX-GDDP-CMIP6](https://www.nccs.nasa.gov/services/data-collections/land-based-products/nex-gddp-cmip6) — daily downscaled climate projections, publicly available on `s3://nex-gddp-cmip6` (anonymous access, `us-west-2`).

Based on the [VirtualiZarr usage docs](https://virtualizarr.readthedocs.io/en/stable/usage.html).

In [ ]:
import warnings
import zarr
import xarray as xr
import icechunk
from obstore.store import from_url
from virtualizarr import open_virtual_dataset, open_virtual_mfdataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning,
)

# Limit concurrent S3 chunk requests. Lower values are more reliable on slow
# or high-latency connections (e.g. laptop on wifi); increase when running
# close to the bucket (e.g. on AWS us-west-2) for better throughput.

In [ ]:
#import zarr
#zarr.config.defaults

In [ ]:
#zarr.config.set({"async.concurrency": 4})

## Setup: S3 store and registry

We point `obstore` at the public S3 bucket and register it so VirtualiZarr knows where to resolve chunk references.

In [ ]:
bucket = "s3://nex-gddp-cmip6"
base = "NEX-GDDP-CMIP6/ACCESS-CM2/ssp126/r1i1p1f1/tasmax"

url_2015 = f"{bucket}/{base}/tasmax_day_ACCESS-CM2_ssp126_r1i1p1f1_gn_2015_v2.0.nc"
url_2016 = f"{bucket}/{base}/tasmax_day_ACCESS-CM2_ssp126_r1i1p1f1_gn_2016_v2.0.nc"

store = from_url(bucket, region="us-west-2", skip_signature=True)
registry = ObjectStoreRegistry({bucket: store})
parser = HDFParser()

In [ ]:
url_2015

## 1. Open a single virtual dataset

`open_virtual_dataset` reads only the metadata (no chunk data is downloaded). The result looks like an xarray dataset but chunk references point back to the original S3 file.

In [ ]:
vds = open_virtual_dataset(
    url=url_2015,
    parser=parser,
    registry=registry,
    loadable_variables=["time", "lat", "lon"],
    decode_times=True,
)
vds

The `ManifestArray` objects show where each chunk lives — no data has been read yet:

In [ ]:
vds["tasmax"].data

## 2. Combine multiple files

To combine several files we open each one as a virtual dataset, then concatenate the list — no data is read, only the chunk manifests are joined.

In [ ]:
urls = [url_2015, url_2016]

vds_list = [
    open_virtual_dataset(
        url=url,
        parser=parser,
        registry=registry,
        loadable_variables=["time", "lat", "lon"],
        decode_times=True,
    )
    for url in urls
]

In [ ]:
vds_list

In [ ]:
combined_vds = xr.concat(vds_list, dim="time", coords="minimal", compat="override")
combined_vds

`open_virtual_mfdataset` does the same thing in one call — equivalent to `xr.open_mfdataset` but producing virtual references instead of loading data:

In [ ]:
combined_vds = open_virtual_mfdataset(
    [url_2015, url_2016],
    registry=registry,
    parser=parser,
    combine="by_coords",
    combine_attrs="drop_conflicts",
    loadable_variables=["time", "lat", "lon"],
    decode_times=True,
)
combined_vds

In [ ]:
combined_vds.vz.to_kerchunk("combined.json", format="json")

## 3. Write to Icechunk

We create a local Icechunk repository on disk and write the virtual references into it. The `VirtualChunkContainer` tells Icechunk where to fetch actual chunk bytes at read time — the original S3 files are never copied.

After running this cell you can browse the `nex-gddp-icechunk/` folder and see the metadata Icechunk stores to track versions and chunk references.

In [ ]:
import shutil
from pathlib import Path

repo_path = Path("nex-gddp-icechunk")
if repo_path.exists():
    shutil.rmtree(repo_path)
    print(f"Cleared existing repo at {repo_path}/")


In [ ]:
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix="s3://nex-gddp-cmip6/",
        store=icechunk.s3_store(region="us-west-2", anonymous=True),
    ),
)

storage = icechunk.local_filesystem_storage(str(repo_path))
repo = icechunk.Repository.create(storage, config)

session = repo.writable_session("main")
vds_list[0].vz.to_icechunk(session.store)
snapshot_id = session.commit("Add 2015 tasmax")
print("Committed:", snapshot_id)

In [ ]:
session = repo.writable_session("main")
vds_list[1].vz.to_icechunk(session.store, append_dim="time")
snapshot_id = session.commit("Append 2016 tasmax")
print("Committed:", snapshot_id)

## 4. Verify: read back from Icechunk

Open the store with xarray — now both years appear as a single continuous dataset. Chunk data is fetched lazily from S3 on demand.

In [ ]:
from pathlib import Path
repo_path = Path("nex-gddp-icechunk")
storage = icechunk.local_filesystem_storage(str(repo_path))

In [ ]:
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix="s3://nex-gddp-cmip6/",
        store=icechunk.s3_store(region="us-west-2", anonymous=True),
    ),
)

In [ ]:
credentials = icechunk.containers_credentials({
    "s3://nex-gddp-cmip6/": icechunk.s3_credentials(anonymous=True)
})

repo2 = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=credentials)
session = repo2.readonly_session("main")

# chunks=None disables dask, avoiding async runtime conflicts with icechunk
ds = xr.open_zarr(session.store, consolidated=False, chunks=None)
ds

In [ ]:
# Fetch the first day and plot a map (triggers actual S3 reads)
ds["tasmax"].isel(time=-1).plot()

The dataset spans both files seamlessly. Here we compute a global mean time series across the last month of 2015 and all of 2016, reading virtual chunks from both original NetCDF files on the fly:

In [ ]:
ds["tasmax"].sel(time=slice("2015-12", "2016-01")).mean(['lon','lat']).plot()